# Part 6: Evaluating a Reranker

This guide provides instructions for the next phase of optimising your RAG pipeline: adding and evaluating a **reranker**. A reranker is a powerful component that can significantly improve the quality of the information your chatbot uses to form its answers.

> This notebook is a guide to enhancing your project. **Do not run the code cells here directly.** Follow the instructions in your local development environment.

---
## 1.&nbsp; What is a Reranker?

So far, our RAG system uses a single-stage process for finding information: it retrieves the `top_k` most similar chunks from the vector store and sends them directly to the LLM. This is fast and effective, but we can improve its precision.

A reranker introduces a second, more sophisticated stage to this process.

1.  **Stage 1: Initial Retrieval (Fast & Broad)**: The vector store quickly retrieves a larger set of potentially relevant documents (e.g., the top 10 or 20). This initial search is optimised for speed.

2.  **Stage 2: Reranking (Slow & Precise)**: The reranker then takes this larger set of documents and uses a more powerful model (typically a cross-encoder) to carefully re-assess and re-order them based on their actual relevance to the question. It then passes only the best few (e.g., the top 2 or 5) to the LLM.

The key benefit is a cleaner, more relevant context for the LLM. By filtering out the noise from the initial retrieval, the reranker helps reduce hallucinations and leads to more accurate, focused answers.

---
## 2.&nbsp; Preparing for the Experiment

To test the effect of a reranker, we'll add a new evaluation stage. This involves adding new configurations and a new evaluation function to our framework.

### 2.1 Add Reranker Configurations

First, we need to define the parameters for our reranker experiment.

1.  Open `evaluation/evaluation_config.py` in your VSCode editor.
2.  Add the following code to the end of the file.

In [ ]:
# --- Cross-encoder Model for Reranking ---
RERANKER_MODEL_NAME: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# --- Configuration for Reranking Evaluation ---
RERANKER_CONFIGS: list[dict[str, int]] = [
    {'retriever_k': 10, 'reranker_n': 2},
    {'retriever_k': 10, 'reranker_n': 5},
    {'retriever_k': 20, 'reranker_n': 5},
]

* `RERANKER_MODEL_NAME`: This defines the specific cross-encoder model we'll use for the precision reranking step.
* `RERANKER_CONFIGS`: This list defines our experiments. `retriever_k` is the number of documents we initially fetch, and `reranker_n` is the number of documents we pass to the LLM after reranking.

> **Why are we keeping the evaluations separate?**
>
> You might notice that the reranker's performance could be influenced by the chunking strategy. While true, testing every combination would create a huge number of experiments, quickly using up your free API credits. By "siloing" the evaluations, we test one variable at a time. This is a pragmatic approach to find improvements without incurring high costs.

### 2.2 Create the Reranker Evaluation Function

Now, let's update `evaluation/evaluation_engine.py` with the logic to test our reranker configurations.

1.  **Add New Imports**: Open `evaluation/evaluation_engine.py` and add the new imports. The `RetrieverQueryEngine` and `SentenceTransformerRerank` are new tools for this stage.

In [ ]:
# Add to existing llama-index.core imports
from llama_index.core.query_engine import RetrieverQueryEngine # <-- Add this line
from llama_index.core.postprocessor import SentenceTransformerRerank # <-- Add this line

# Add the new configs to the import from evaluation.evaluation_config
from evaluation.evaluation_config import (
    # ... existing imports
    CHUNKING_STRATEGY_CONFIGS,
    RERANKER_MODEL_NAME, # <-- Add this line
    RERANKER_CONFIGS, # <-- Add this line
)

2.  **Add the Evaluation Function**: Add the following function to the file, placing it after the `evaluate_chunking_strategies` function.

In [ ]:
def evaluate_reranker_strategies() -> None:
    """
    Evaluates different reranker settings on top of the best chunking strategy.
    """
    print("\n--- 🚀 Stage 3: Evaluating Reranker Strategies ---")

    llm_to_test: Groq = initialise_llm()

    embed_model_to_test: HuggingFaceEmbedding = get_embedding_model()

    questions, ground_truths = get_evaluation_data()

    ragas_llm: LlamaIndexLLMWrapper
    ragas_embeddings: HuggingFaceEmbeddings
    ragas_llm, ragas_embeddings = load_ragas_models()

    index: VectorStoreIndex = get_or_build_index(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        embed_model=embed_model_to_test
    )

    all_results: list[pd.DataFrame] = []

    for config in RERANKER_CONFIGS:

        retriever_k, reranker_n = config['retriever_k'], config['reranker_n']

        print(f"--- Testing Reranker Config: retrieve_k={retriever_k},"
              f" rerank_n={reranker_n} ---")

        retriever = index.as_retriever(similarity_top_k=retriever_k)

        reranker = SentenceTransformerRerank(
            top_n=reranker_n, model=RERANKER_MODEL_NAME
        )

        query_engine = RetrieverQueryEngine.from_args(
            retriever=retriever,
            node_postprocessors=[reranker],
            llm=llm_to_test
        )

        qa_dataset: Dataset = generate_qa_dataset(
            query_engine,
            questions,
            ground_truths
        )

        print("--- Running Ragas evaluation for reranker... ---")

        # --- If you don't have a Rate per Minute limit on your API ---
        # results_df: pd.DataFrame = evaluate_without_rate_limit(
        #     qa_dataset,
        #     ragas_llm,
        #     ragas_embeddings,
        # )

        # --- If you do have a Rate per Minute API limit ---
        results_df: pd.DataFrame = evaluate_with_rate_limit(
            qa_dataset,
            ragas_llm,
            ragas_embeddings,
        )

        results_df['chunk_size'] = CHUNK_SIZE
        results_df['chunk_overlap'] = CHUNK_OVERLAP
        results_df['retriever_k'] = retriever_k
        results_df['reranker_n'] = reranker_n

        all_results.append(results_df)

    final_df: pd.DataFrame = pd.concat(all_results, ignore_index=True)

    save_results(final_df, "reranker_evaluation")

    print("--- ✅ Reranker Strategy Evaluation Complete ---")

> **Why the `RetrieverQueryEngine`?**
>
> You'll notice we've switched from the simple `index.as_query_engine()` to the more explicit `RetrieverQueryEngine.from_args()`. This is necessary because a reranker is a type of `node_postprocessor` - a component that runs after the initial retrieval but before the final answer generation. The `RetrieverQueryEngine` gives us the fine-grained control to construct this multi-stage pipeline by explicitly defining the `retriever` and the list of `node_postprocessors`.

### 2.3 Update the Main Execution Block

Finally, let's update the main execution block to run our new stage.

In `evaluate.py`, update the `if __name__ == "__main__":` block:

In [ ]:
from evaluation.evaluation_engine import (
    # evaluate_baseline,
    # evaluate_chunking_strategies,
    evaluate_reranker_strategies,
)


if __name__ == "__main__":

    # Run Stage 1: Baseline Evaluation
    # evaluate_baseline()

    # Run Stage 2: Chunking Strategy Evaluation
    # evaluate_chunking_strategies()

    # Run Stage 3: Reranker Strategy Evaluation
    evaluate_reranker_strategies()

---
## 3.&nbsp; Run the Reranker Evaluation

With the new function and configurations in place, you are ready to run the experiment.

From your VSCode terminal, run the evaluation script as before:

In [ ]:
python evaluate.py

The first time you run this, it will download the reranker model, which may take a moment. Since you've commented out the previous stages, the script will proceed directly to the reranker evaluation.

---
## 4.&nbsp; Analyse the Results

Once the script finishes, navigate to the `evaluation/evaluation_results` folder. You will find the new `reranker_evaluation_summary_... .csv` file.

Open this summary file. Your goal is to see how the reranker impacts performance. Does increasing the number of documents retrieved (`retriever_k`) and then filtering them down with the reranker (`reranker_n`) improve the final scores? Pay close attention to **`context_precision`**, as this metric should be most directly improved by a good reranker.

---
## 5.&nbsp; Challenges and Further Exploration 😀

You now have a complete, three-stage evaluation framework. The final step is to apply these techniques to your own project and then integrate the best-performing components into your main chatbot.

### 1. Find Your Optimal Reranker Strategy

It's time to run the reranker evaluation on your custom RAG system and analyse the results to find the best settings for your specific documents.

**Your Mission:**

1.  **Confirm Your Baseline**: Double-check that `CHUNK_SIZE` and `CHUNK_OVERLAP` in `src/config.py` is set to the optimal values you found for your project.
2.  **Run the Reranker Evaluation**: Execute `python evaluate.py`.
3.  **Set Up Your Analysis Notebook**: In your `notebooks` directory, create a new Notebook file named `03_reranker_analysis.ipynb`.
4.  **Load and Analyse Your Results**:
    * In the new notebook, load the summary results from your reranker experiment.
    * Analyse the results. Which configuration of `retriever_k` and `reranker_n` gives the best scores? Does the reranker provide a significant improvement over your best chunking-only strategy?
    * Document your conclusions in the notebook with text and plots.

### 2. Deeper Experimentation & Final Integration

Now that you have a feel for the process, you can conduct more advanced experiments.

**Challenge:**

* **Experiment with `retriever_k`**: How does retrieving many more documents initially (e.g., `retriever_k: 50`) affect the reranker's performance and the final scores? Is there a point of diminishing returns?
* **Try Different Reranker Models**: The `cross-encoder/ms-marco-MiniLM-L-6-v2` model is small and fast. You can find more powerful (but slower) models on the Hugging Face Hub. Try swapping one in and see if it improves your scores.


---
## 5.&nbsp; Implement the Ranker into your RAG Pipeline (Optional)

You've sucessfully performed some advanced experiments and concluded that a reranker improves your chatbot, as well as the top combination of `retriever_k`and `reranker_n`. Now it's time to implement this into our main application!


### 5.1 Update the application with your chosen configuration

Add the new constants for your reranker.

1. **Add New Constants:** open `src/config.py` and add the code below


In [ ]:
# --- Reranker Configuration ---
RERANKER_TOP_N = 3  # For example, The number of nodes to return after reranking
RERANKER_MODEL_NAME = "BAAI/bge-reranker-base" # Or another cross-encoder model

2. **Add imports:** open `src/engine.py` and add the code below

In [ ]:
from llama_index.core.chat_engine import CondensePlusContextChatEngine
from llama_index.core.postprocessor import SentenceTransformerRerank

# Add these to your config imports:
from src.config import (
    # ... previous imports
    RERANKER_TOP_N,
    RERANKER_MODEL_NAME,
)

### 5.2 Upgrade the Chat Engine

we now want to update our `BaseChatEngine`

Replacing the generic `BaseChatEngine` with the `CondensePlusContextChatEngine` and a `SentenceTransformerRerank` postprocessor is a professional upgrade that shifts your system from a "simple chatbot" to an "advanced RAG pipeline".

The core reason for this change is that a simple `BaseChatEngine` often struggles with conversation history and retrieval accuracy as a chat progresses.



1. **update `get_chat_engine`:** in `src/engine.py` and replace the function `get_chat_engine` with the new code below

In [ ]:
def get_chat_engine(
        llm: Groq,
        embed_model: HuggingFaceEmbedding
) -> CondensePlusContextChatEngine:
    """Initialises and returns the main conversational RAG chat engine."""
    vector_index: VectorStoreIndex = get_vector_store(embed_model)

    # 1. Create the Retriever
    retriever = vector_index.as_retriever(similarity_top_k=SIMILARITY_TOP_K)

    # 2. Setup the Reranker
    reranker: SentenceTransformerRerank = SentenceTransformerRerank(
        top_n=RERANKER_TOP_N,
        model=RERANKER_MODEL_NAME
    )

    # 3. Setup Memory
    memory: ChatMemoryBuffer = ChatMemoryBuffer.from_defaults(
        token_limit=CHAT_MEMORY_TOKEN_LIMIT
    )

    # 4. Assemble the advanced Chat Engine
    chat_engine = CondensePlusContextChatEngine(
        retriever=retriever,
        llm=llm,
        memory=memory,
        system_prompt=LLM_SYSTEM_PROMPT,
        node_postprocessors=[reranker]
    )

    return chat_engine

**Why this upgrade matters?**

**Standalone Queries**: By condensing history into a single question, the retriever is much more likely to find relevant documents than if it queried with the whole chat history.

**Reranking Precision**: Retrieval using embeddings (like bge-large) is good at finding "similar" chunks, but the SentenceTransformerRerank uses a more expensive Cross-Encoder to verify which of those chunks is truly relevant to the query, moving the best info to the top.

**Memory Management**: The `CondensePlusContextChatEngine` maintains state, allowing it to answer meta-questions like "What was the previous topic we discussed?" effectively.

2. **update main_chat_loop:** in `src/engine.py` replace `BaseChatEngine` inside function `main_chat_loop` with our updated `CondensePlusContextChatEngine`

In [ ]:
def main_chat_loop() -> None:
    """Main application loop to run the RAG chatbot."""
    print("--- Initialising models... ---")
    llm: Groq = initialise_llm()
    embed_model: HuggingFaceEmbedding = get_embedding_model()

    chat_engine: CondensePlusContextChatEngine = get_chat_engine(
        llm=llm,
        embed_model=embed_model
    )
    print("--- RAG Chatbot Initialised. ---")
    chat_engine.chat_repl()

### 5.3 Run the main application

In [ ]:
python main.py

Congratulations! You have sucessfully implemented a Rag Pipeline with rerank and advanced chat options. Play around with different questions to test the output